In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By  
from selenium.webdriver.support.ui import Select  
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
import os
import re

### 사용자 정의 함수 

In [7]:
# 전처리 함수
def preprocess_text(text):
    # 문자열이 아닐 경우, 문자열로 변환하거나 빈 문자열로 처리
    if not isinstance(text, str):
        text = str(text)  # 또는 text = ""
    
    # 특수기호 제거
    text = re.sub(r'[※ㅇ\-•●◆▶★]', '', text)

    # 한자 제거
    text = re.sub(r'\(([\u4E00-\u9FFF]{1,10})\)', '', text)

    # 여러 줄을 한 문장 단위로 정리
    lines = text.strip().splitlines()
    lines = [line.strip() for line in lines if line.strip()]  # 공백 제거 + 빈 줄 제거

    # 문장 내 연속 공백 제거
    lines = [re.sub(r'\s+', ' ', line) for line in lines]

    # 리스트를 하나의 문자열로 합치기 (여기서는 공백을 구분자로 사용)
    return " ".join(lines)



In [2]:
# 사이트 불러오기
driver = webdriver.Chrome(service = Service(), options = webdriver.ChromeOptions())  
driver.get('https://unipass.customs.go.kr/clip/index.do')


In [3]:
TIME = 3 

driver.find_element(By.ID, 'TOPMENU_LNK_M_ULS0200000000').click()  # HS CODE 버튼
time.sleep(TIME)

# 국내 품목 사례 버튼 
driver.find_element(By.ID, 'LEFTMENU_LK_M_ULS0807030051').click() 
time.sleep(TIME)

In [13]:
# import pandas as pd

# l = [] 
# j = '1.2 2.3 3.4 4.5 5.6 6.7 7.8'
# # l.append('a')
# # l.append()
# l.append(j.split(' ')[1:6:2]) # HS CODE 숫자만 
# l

# # 데이터 프레임 생성
# df = pd.DataFrame({
#     'values': [['123.241-2', '123.532-2']]
# })

# # df['values'].apply(lambda x: re.sub(r'[^0-9]', '', x)) # 리스트가 담긴 값 처리 X 

# import re

# text = "123.456-34"
# only_numbers = re.sub(r'\D', '', text)  # \D는 숫자가 아닌 문자
# print(only_numbers)

                   values
0  [123.241-2, 123.532-2]


In [ ]:
# 데이터 저장 형태 : 리스트? 딕셔너리? 
TIME = 5
date_list = []     # 시행 일자 
agency_list = []   # 시행 기관 
hs_code_list = []  # hs code  
product_list = []  # 품명 
pd_description_list = []   # 물품 설명 
hs_description_list = []   # 결정 사유
count = 0
hs_code_samples = {} 

Ryu = [str(i).zfill(2) for i in range(78, 98)]

# 류 입력 
    ele = driver.find_element(By.ID, 'srchDtrmHsSgn')
    ele.send_keys(Ryu)
    ele.send_keys(Keys.ENTER)
    time.sleep(TIME)
    
    
    while True:
        # 10페이지씩 사례 분석
        for i in range(1,11): 
            # 다음 페이지 클릭
            # 1 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[1]/li[3]/a 
            # 2 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[2]/li[3]/a
            # 3 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[2]/li[3]/a
            driver.find_element(By.XPATH, f'//*[@id="ULS0203042S_T3_container"]/div[2]/ul[{count + 1}]/li[{i}]').click()
    
            time.sleep(TIME)
                         
            for i in range(1,11):
                # 사례 클릭 
                driver.find_element(By.XPATH, f'//*[@id="ULS0203042S_T3_table1"]/tbody/tr[{i}]/td[4]/a').click()
                time.sleep(TIME)
                # HS CODE가 여러개일 경우 .. 처리할 방법 
                      
                # 사례 정보 수집 (append()는 null값은 에러가 뜨는 것 같다. )
                      
                date_list.append(driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[2]/td').text)
                agency_list.append(driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[3]/td').text)
                hs_code = driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[4]/td').text
                print(hs_code)
                
                # 결정 세번이 여러개인 경우
                if len(hs_code) > 1:
                    n = len(hs_code)
                    # hs_code_list.append(hs_code.split(' ')[1:n:2].apply(lambda x: re.sub(r'[^0-9]', '', x))) # HS CODE 숫자만 남기기
                    # apply() 는 데이터 프레임이나 시리즈에서만 사용 가능 
                    hs_code_list.append(hs_code.split(' ')[1:n:2])
                else:
                    hs_code_list.append(re.sub(r'\D', '', hs_code.split(' ')[0]))  # HS CODE 숫자만 남기기
                
                product_list.append(driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[5]/td').text)
                pd_description_list.append(driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[6]/td').text)
                hs_description_list.append(driver.find_element(By.XPATH, f'//*[@id="ULS0203037S_T1_table1"]/tbody/tr[7]/td').text)
                       
        if count == 1: 
            driver.find_element(By.XPATH, '//*[@id="ULS0203042S_T3_container"]/div[2]/ul[3]/li[1]/a').click()
            time.sleep(TIME)
        if count == 0:
            driver.find_element(By.XPATH, '//*[@id="ULS0203042S_T3_container"]/div[2]/ul[2]/li[1]/a').click()
            time.sleep(TIME)
            count += 1       
                      
    # print(count(hs_code_list))
    print(len(hs_code_list))
    
    # 1 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[1]/li[3]/a 
    # 2 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[2]/li[3]/a
    # 3 페이지 3번 //*[@id="ULS0203042S_T3_container"]/div[2]/ul[2]/li[3]/a
    
    # 데이터 프레임에 저장 
    hs_code_samples['date'] = date_list
    hs_code_samples['agency'] = agency_list
    hs_code_samples['hs_code'] = hs_code_list
    hs_code_samples['product'] = product_list
    hs_code_samples['pd_description'] = pd_description_list
    hs_code_samples['hs_description'] = hs_description_list


In [ ]:
# 데이터 프레임 변환
import pandas as pd          

df = pd.DataFrame(hs_code_samples)
display(df.head())

# 기본 문자 전처리 
columns = df.columns

for col in columns:
    df[col] = df[col].apply(preprocess_text)

# 자료 저장 
df.to_csv("hs_code_samples.csv", index=False)

In [ ]:
# # 텍스트 파일 저장
# with open('./datasets/hs_code_samples.txt','w',encoding='utf-8') as f:
#     f.write(date_list)
# f.close()
# print('End')


In [5]:
import pandas as pd 

df = pd.read_csv('./datasets/hs_code_samples.csv')

In [11]:
hs_33.head(1)
hs_33['hs_description'][0]
hs_33['pd_description'][0]

'방향성물질(계피 정유 5%, 오레가노 정유 1%), 메도우스위트 추출물 5%, 마늘 추출물 0.1%, 정제수로 조성된 암갈색 액상\n- 용도: 사료용(음수에 첨가하여 급여)\n※ 품목분류는 수출입신고 당시의 물품상태에 따라 변경될 수 있음'

In [9]:
import re
columns = df.columns
for col in columns:
    df[col] = df[col].apply(preprocess_text)
df.head(1)

,date,agency,hs_code,product,pd_description,hs_description
0,20250207,관세평가분류원,3302.900000,Mixture with a basis of odoriferous substances...,"방향성물질(계피 정유 5%, 오레가노 정유 1%), 메도우스위트 추출물 5%, 마늘...",관세율표 제33류 주 제2호에 “제3302호에서 “방향성 물질”이란 제3301호의 ...
